# TakeMeter — Fine-Tuning Notebook (football / soccer discourse)
### AI201 · Project 3

This notebook fine-tunes a text classifier on your annotated dataset and compares it
to a zero-shot baseline.

**What this notebook does for you (infrastructure):**
- Tokenizes your dataset and prepares it for training
- Fine-tunes DistilBERT with **fp16 mixed precision** on the free T4 GPU
- Computes evaluation metrics (accuracy + macro-F1 + per-class) and a confusion matrix
- Runs the Groq baseline **in parallel with joblib** and compares both models
- **Exports the fine-tuned model** (Section 7) so the `web/` app can serve live predictions

**What you do (the actual work):**
- Collect and annotate your 200+ examples (use the web **Train** page + the model prompts in `prompts/`)
- Confirm your label map and upload your labeled CSV
- The Groq prompt is pre-filled with the soccer taxonomy — review it against your `planning.md`
- Analyze the output and write your evaluation report

**Labels:** `analysis` · `hot_take` · `reaction` · `mixed` (defined in `taxonomy.json`).

---
**Before you start:** set the runtime to **T4 GPU** — *Runtime → Change runtime type → T4 GPU → Save*.
The notebook auto-detects the GPU and enables fp16; without it, training falls back to (slow) CPU.

In [ ]:
# Install any dependencies not pre-installed on Colab
# joblib ships with Colab but we pin it here so the parallel baseline always runs.
!pip install -q groq python-dotenv joblib
print("✅ Dependencies ready")

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset
from joblib import Parallel, delayed   # ← parallelizes the Groq baseline (Section 5)
import warnings
warnings.filterwarnings("ignore")

# ── Runtime / hardware setup ──────────────────────────────────────────────
# DEVICE drives fp16 mixed-precision training (Section 3). fp16 only works on a
# CUDA GPU, so we gate on it — set your runtime to **T4 GPU** for the speed-up.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = DEVICE == "cuda"          # mixed precision: ~1.5–2x faster on a T4
N_JOBS = 8                           # parallel threads for the Groq baseline calls

print("✅ Imports complete")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Mixed precision (fp16): {USE_FP16}")
else:
    print("⚠️  No GPU detected — set Runtime → Change runtime type → T4 GPU for fast training.")

---
## Section 1: Load Your Dataset

Upload your labeled CSV and define your label map.  
Your CSV must have at least two columns: `text` (the post/comment) and `label` (your string label).

In [ ]:
# ── TakeMeter label map (football / soccer discourse) ─────────────────────
# Single source of truth: ../taxonomy.json in the repo. Four trainable labels.
#   analysis  — structured argument with specific, verifiable evidence
#   hot_take  — bold, confident opinion asserted without real evidence
#   reaction  — in-the-moment emotional response to a recent event
#   mixed     — genuine blend where no single label dominates
#
# NOTE: "popular" was dropped (it depends on engagement counts the text model
# can't see) and "skip"/"unlabeled" rows are filtered out below — they are a
# labeling escape hatch, not a training class.
# ────────────────────────────────────────────────────────────────────────

LABEL_MAP = {
    "analysis":  0,
    "hot_take":  1,
    "reaction":  2,
    "mixed":     3,
}

# ── END ────────────────────────────────────────────────────────────────────

ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
NUM_LABELS = len(LABEL_MAP)
print(f"Labels: {LABEL_MAP}")
print(f"Number of labels: {NUM_LABELS}")

In [ ]:
# Upload your CSV from your computer
from google.colab import files
print("Select your labeled dataset CSV file...")
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {CSV_PATH}")

In [ ]:
# Load and validate your dataset
df = pd.read_csv(CSV_PATH)

# ── TODO (if needed) ──────────────────────────────────────────────────────
# If your CSV uses different column names, rename them here.
# Example: df = df.rename(columns={"post": "text", "category": "label"})
# ── END TODO ──────────────────────────────────────────────────────────────

df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip().str.lower()

# Drop labeling-only / unlabeled rows — these are not training classes.
NON_TRAINING = {"skip", "unlabeled", "popular", "", "nan"}
before = len(df)
df = df[~df["label"].isin(NON_TRAINING)].copy()
dropped = before - len(df)
if dropped:
    print(f"Dropped {dropped} non-training rows (skip/unlabeled/blank/popular).")

print(f"Columns: {df.columns.tolist()}")
print(f"Total trainable examples: {len(df)}")
print()
print("Label distribution:")
print(df["label"].value_counts())

# Validate all remaining labels are in LABEL_MAP
unknown = set(df["label"].unique()) - set(LABEL_MAP.keys())
if unknown:
    print(f"\n⚠️  Labels in CSV not found in LABEL_MAP: {unknown}")
    print("Update your LABEL_MAP above (or fix the CSV) so every label maps.")
else:
    print("\n✅ All labels match your LABEL_MAP")

# Imbalance guardrail (spec: no class > 70% of the dataset)
top_share = df["label"].value_counts(normalize=True).max()
if top_share > 0.70:
    print(f"⚠️  Most common label is {top_share:.0%} of the data (>70%). "
          "Collect more of the minority classes before training.")

# Convert string labels to integers
df["label_id"] = df["label"].map(LABEL_MAP)
df = df.dropna(subset=["label_id"])
df["label_id"] = df["label_id"].astype(int)

---
## Section 2: Prepare Data for Training

Splits your dataset into train / validation / test sets and tokenizes the text.

In [ ]:
# Train / val / test split — 70% / 15% / 15%
# Stratified so each split has roughly the same label distribution.
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["label_id"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["label_id"]
)

print(f"Train: {len(train_df)} examples")
print(f"Validation: {len(val_df)} examples")
print(f"Test: {len(test_df)} examples")
print()
print("Train label distribution:")
print(train_df["label"].value_counts())
print()
print("Test label distribution:")
print(test_df["label"].value_counts())

# Reset indices (needed for clean HuggingFace Dataset conversion)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

In [ ]:
# Load tokenizer and tokenize all splits
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

def make_dataset(df_split):
    ds = Dataset.from_pandas(
        df_split[["text", "label_id"]].rename(columns={"label_id": "labels"})
    )
    return ds.map(tokenize, batched=True)

train_dataset = make_dataset(train_df)
val_dataset   = make_dataset(val_df)
test_dataset  = make_dataset(test_df)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("✅ Tokenization complete")
print(f"Sample keys: {list(train_dataset[0].keys())}")

---
## Section 3: Fine-Tune Your Model

Loads `distilbert-base-uncased` with a classification head and fine-tunes it on your training data.  
Training runs for 3 epochs and takes **5–15 minutes** on a T4 GPU.

> **Hyperparameter note:** The defaults below work well for datasets of 100–500 examples.  
> If you change any values, note what you changed and why in your README.

In [ ]:
# Load DistilBERT with a classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_MAP,
)
print(f"✅ Model loaded: {MODEL_NAME}")
print(f"Output labels: {NUM_LABELS}")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        # macro-F1 weights every class equally — the right summary metric for an
        # imbalanced, multi-class task where minority classes matter.
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
    }

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────
# num_train_epochs  — passes through the training data; 3 is a good default
#                     for small datasets. Increase cautiously; more epochs
#                     risk overfitting on ~200 examples.
# learning_rate     — 2e-5 is the standard starting point for fine-tuning
#                     BERT-family models. Lower → slower but more stable.
# per_device_train_batch_size — 16 fits T4 GPU comfortably.
#                     Reduce to 8 if you get out-of-memory errors.
# fp16              — mixed precision. On a T4 this ~halves training time and
#                     memory with no accuracy loss. Auto-disabled off-GPU.
# dataloader_num_workers — parallel CPU data loading (Colab gives 2 vCPUs).
# ─────────────────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./takemeter-model",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=50,
    fp16=USE_FP16,                 # ← T4 mixed-precision speed-up
    dataloader_num_workers=2,      # ← parallel data loading
    dataloader_pin_memory=USE_FP16,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",   # pick the checkpoint with best macro-F1
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Starting fine-tuning on {DEVICE} (fp16={USE_FP16})... (5–15 min on T4 GPU)")
trainer.train()
print("\n✅ Fine-tuning complete")

---
## Section 4: Evaluate Fine-Tuned Model on Test Set

Runs inference on your locked test set and generates metrics and a confusion matrix.  
These numbers go directly into your evaluation report.

In [ ]:
# Run inference on the test set
print("Running inference on test set...")
ft_output = trainer.predict(test_dataset)
ft_pred_ids = np.argmax(ft_output.predictions, axis=-1)
ft_true_ids = ft_output.label_ids

ft_probs = torch.nn.functional.softmax(
    torch.tensor(ft_output.predictions), dim=-1
).numpy()

# Overall accuracy
ft_accuracy = accuracy_score(ft_true_ids, ft_pred_ids)
print(f"\n🎯 Fine-tuned model accuracy: {ft_accuracy:.3f}")

# Per-class metrics
label_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
print("\nPer-class metrics (fine-tuned model):")
print(classification_report(ft_true_ids, ft_pred_ids, target_names=label_names, zero_division=0))

In [ ]:
# Confusion matrix
cm = confusion_matrix(ft_true_ids, ft_pred_ids)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Fine-Tuned Model — Confusion Matrix (Test Set)")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("✅ Saved: confusion_matrix.png  →  commit this to your repo and include in README")

In [ ]:
# Print wrong predictions for your error analysis
# Review these carefully — pick 3 to analyze in depth in your README.

wrong_idx = np.where(ft_pred_ids != ft_true_ids)[0]
print(f"Wrong predictions: {len(wrong_idx)} / {len(ft_true_ids)}\n")

for i, idx in enumerate(wrong_idx[:15]):
    text = test_df.iloc[idx]["text"]
    true_label = ID_TO_LABEL[ft_true_ids[idx]]
    pred_label = ID_TO_LABEL[ft_pred_ids[idx]]
    confidence = ft_probs[idx][ft_pred_ids[idx]]
    print(f"--- #{i+1} ---")
    print(f"Text:      {text[:200]}{'...' if len(text) > 200 else ''}")
    print(f"True:      {true_label}")
    print(f"Predicted: {pred_label}  (confidence: {confidence:.2f})")
    print()

---
## Section 5: Baseline Classifier (Groq)

Runs your zero-shot baseline using `llama-3.3-70b-versatile`.  
You need to write the classification prompt using your label definitions.

In [ ]:
from groq import Groq

# ── TODO: Add your Groq API key ───────────────────────────────────────────
# Recommended: use Colab Secrets so your key is never visible in the notebook.
#   1. Click the 🔑 icon in the left sidebar ("Secrets")
#   2. Add a secret named GROQ_API_KEY with your key as the value
#   3. Enable notebook access for the secret
#
# Then uncomment Option A below (and delete Option B).
#
# Option A — Colab Secrets (recommended):
from google.colab import userdata
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
#
# Option B — paste directly (do not commit to GitHub):
#GROQ_API_KEY = "your_groq_api_key_here"

assert GROQ_API_KEY, (
    "GROQ_API_KEY not set — add it in the Colab Secrets panel (\U0001f511, left "
    "sidebar) and enable notebook access for this notebook, or use Option B above."
)

client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq client initialized")

In [ ]:
# ── Groq zero-shot classification prompt (football / soccer discourse) ─────
# Definitions copied verbatim from taxonomy.json / planning.md so the baseline is
# judged against the SAME boundary as the fine-tuned model. Full write-up:
# prompts/groq_baseline_prompt.md. The final line forces a clean single-token
# answer that classify_with_groq() can parse.
# ─────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """
You are classifying posts from a football (soccer) community — r/soccer-style
discussion about the World Cup, MLS, and clubs. Assign each post to EXACTLY ONE of
the four categories below.

analysis: A structured argument backed by specific, verifiable evidence —
statistics, historical comparison, or tactical observation. It reasons toward a
conclusion instead of just asserting one.
Example: "City's PPDA dropped from 11.2 to 8.7 because Rodri drops between the
center-backs and both fullbacks invert into midfield."

hot_take: A bold, confident opinion stated WITHOUT real supporting evidence. The
claim might be true, but the post asserts rather than argues; often contrarian.
Example: "Pep is overrated, anyone could win with that budget. Change my mind."

reaction: An immediate emotional response to a specific recent event. Little to no
argument — the post is expressing a feeling in the moment.
Example: "97th-minute winner in the derby, I AM SHAKING, I cannot breathe right now."

mixed: A genuine combination of the above — for example an emotional reaction that
ALSO carries a real argument — where no single category clearly dominates.
Example: "Gutted we lost but the xG was 0.4 to 2.1, we got battered, same broken
midfield all season."

Rules:
- If specific, verifiable evidence survives removing the opinion framing -> analysis.
- If the evidence is vague, cherry-picked, or decorative -> hot_take.
- If it is an in-the-moment emotional response with no real argument -> reaction.
- If it truly blends reaction and argument with neither dominant -> mixed.

Respond with ONLY the category name, lowercase, nothing else.
Do not explain. Valid outputs: analysis, hot_take, reaction, mixed
"""

print("Prompt length:", len(SYSTEM_PROMPT), "characters")

In [ ]:
def classify_with_groq(text, max_retries=4):
    """Classify a single post. Returns a label string or None if unparseable.

    Retries with exponential backoff on transient/rate-limit (429) errors so the
    parallel sweep below doesn't lose examples when Groq's free tier throttles us.
    """
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Classify this post:\n\n{text}"},
                ],
                temperature=0,
                max_tokens=20,
            )
            raw = response.choices[0].message.content.strip().lower()
            # Match the model's output to a label. Check longest labels first so a
            # label that is a substring of another can't be matched by mistake.
            for label in sorted(LABEL_MAP, key=len, reverse=True):
                if raw == label or label in raw:
                    return label
            return None  # model output didn't match any known label
        except Exception as e:
            msg = str(e).lower()
            if ("429" in msg or "rate" in msg or "timeout" in msg) and attempt < max_retries - 1:
                time.sleep(2 ** attempt)   # 1s, 2s, 4s backoff
                continue
            print(f"API error (giving up on one example): {e}")
            return None


# ── Run baseline on the test set IN PARALLEL with joblib ──────────────────
# The bottleneck is network latency, not CPU, so a threading backend lets many
# requests be in flight at once. joblib preserves input order, so predictions
# line up with test_df row-for-row. N_JOBS (set in the imports cell) defaults to
# 8 — lower it to 2–4 if you hit sustained 429 rate-limit errors.
texts = test_df["text"].tolist()
print(f"Running baseline on {len(texts)} examples with {N_JOBS} parallel workers...")
t0 = time.time()

baseline_preds = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(classify_with_groq)(t) for t in texts
)

elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s ({elapsed / max(len(texts), 1):.2f}s/example effective)")

none_count = baseline_preds.count(None)
if none_count > 0:
    print(f"\n⚠️  {none_count} responses could not be parsed.")
    if none_count / len(texts) > 0.10:
        print("That's >10% — revise SYSTEM_PROMPT so the model outputs clean label "
              "names, or reduce N_JOBS if these are rate-limit failures.")

In [ ]:
# Baseline metrics (exclude unparseable responses)
valid = [(p, t) for p, t in zip(baseline_preds, test_df["label_id"])
         if p is not None]
bl_pred_ids = [LABEL_MAP[p] for p, _ in valid]
bl_true_ids = [t for _, t in valid]

bl_accuracy = accuracy_score(bl_true_ids, bl_pred_ids)
print(f"🎯 Baseline accuracy: {bl_accuracy:.3f}  "
      f"(evaluated on {len(valid)}/{len(test_df)} parseable responses)")
print()
label_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
print("Per-class metrics (baseline):")
print(classification_report(bl_true_ids, bl_pred_ids, target_names=label_names, zero_division=0))

---
## Section 6: Compare Results and Export

Side-by-side comparison of both models.  
Download the output files and commit them to your GitHub repo.

In [ ]:
ft_macro_f1 = f1_score(ft_true_ids, ft_pred_ids, average="macro", zero_division=0)
bl_macro_f1 = f1_score(bl_true_ids, bl_pred_ids, average="macro", zero_division=0)

print("=" * 56)
print("RESULTS COMPARISON")
print("=" * 56)
print(f"{'Model':<33}{'Accuracy':>10}{'MacroF1':>11}")
print("-" * 56)
print(f"{'Zero-shot baseline (Groq)':<33}{bl_accuracy:>10.3f}{bl_macro_f1:>11.3f}")
print(f"{'Fine-tuned DistilBERT':<33}{ft_accuracy:>10.3f}{ft_macro_f1:>11.3f}")
print("-" * 56)
acc_delta = ft_accuracy - bl_accuracy
f1_delta = ft_macro_f1 - bl_macro_f1
print(f"\nFine-tuning Δaccuracy: {acc_delta:+.3f}   Δmacro-F1: {f1_delta:+.3f}")
print("\nUse these numbers in your README evaluation report.")

In [ ]:
# Save results JSON — commit to your GitHub repo and reference in README
ft_report = classification_report(
    ft_true_ids, ft_pred_ids, target_names=label_names,
    output_dict=True, zero_division=0,
)
bl_report = classification_report(
    bl_true_ids, bl_pred_ids, target_names=label_names,
    output_dict=True, zero_division=0,
)

results = {
    "model": MODEL_NAME,
    "test_set_size": len(test_df),
    "label_map": LABEL_MAP,
    "baseline": {
        "accuracy": round(bl_accuracy, 4),
        "macro_f1": round(bl_report["macro avg"]["f1-score"], 4),
        "per_class_f1": {l: round(bl_report[l]["f1-score"], 4) for l in label_names},
        "parsed": len(valid),
    },
    "finetuned": {
        "accuracy": round(ft_accuracy, 4),
        "macro_f1": round(ft_report["macro avg"]["f1-score"], 4),
        "per_class_f1": {l: round(ft_report[l]["f1-score"], 4) for l in label_names},
    },
    "improvement_accuracy": round(ft_accuracy - bl_accuracy, 4),
    "confusion_matrix": confusion_matrix(ft_true_ids, ft_pred_ids).tolist(),
    "confusion_matrix_labels": label_names,
}
with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("✅ Files ready to download:")
print("   evaluation_results.json  — metrics for your README (now includes per-class F1 + confusion matrix)")
print("   confusion_matrix.png     — include in your README")
print()
print("Download via: Files panel (📁) on the left → right-click → Download")

---
## Section 7: Export the Fine-Tuned Model (for the web app)

Saves the fine-tuned model + tokenizer and zips them so you can download the folder
and drop it into the **web app** (`web/model/`) to power the live classifier on the
Test page. This is what makes the *Deployed Interface* stretch feature work end-to-end.

In [ ]:
# Save the best fine-tuned model + tokenizer to a clean folder, then zip it.
EXPORT_DIR = "takemeter_model_export"
trainer.save_model(EXPORT_DIR)          # saves config + weights (best checkpoint)
tokenizer.save_pretrained(EXPORT_DIR)

# Stash the label map + metrics alongside the weights so the web app is self-contained.
with open(os.path.join(EXPORT_DIR, "takemeter_meta.json"), "w") as f:
    json.dump({"label_map": LABEL_MAP, "id_to_label": ID_TO_LABEL,
               "results": results}, f, indent=2)

# Zip for a one-click download.
import shutil
zip_path = shutil.make_archive("takemeter_model_export", "zip", EXPORT_DIR)
print(f"✅ Model exported and zipped: {zip_path}")
print("\nNext steps for the live web classifier:")
print("  1. Download takemeter_model_export.zip (Files panel 📁 → right-click → Download)")
print("  2. Unzip it into the repo at:  web/model/")
print("  3. Run the web app:  cd web && python app.py  →  open http://127.0.0.1:5000")
print("     The Test page will auto-detect web/model/ and classify with YOUR model.")

# Optional: trigger the browser download directly from Colab.
try:
    from google.colab import files as _files
    _files.download("takemeter_model_export.zip")
except Exception:
    pass